**Legacy research notebook.** Start with `Memact_Baseline.ipynb` for the current measured GPU workflow. This notebook retains experimental features and unverified prototype claims.


# 🎙️ Memact — Google Colab Free GPU Calibration

**Tagline**: *Say it silently.*

This notebook enables fast fine-tuning of **Memact** using a free Cloud GPU (e.g. Nvidia T4 on Google Colab). It solves the hardware barrier of running heavy Conformer backpropagation on low-power ultraportable laptops.

### Workflow:
1. **Record calibration videos** locally using Memact web UI on your laptop.
2. **Export Calibration Package (.zip)** from the Memact UI or CLI (`memact export-calib`).
3. **Upload the zip here** and run this notebook (takes ~30-60 seconds on free T4 GPU).
4. **Download `lora.pt`** back to your laptop and place it in `user_profiles/<username>/lora.pt`.
5. Switch Memact on your laptop to **Personalized Mode** — runs 100% offline and privately!

In [ ]:
# 1. Verify GPU availability
!nvidia-smi

In [ ]:
# Upload memact_eval_source.zip generated with experiments/package_eval_source.py.
from google.colab import files
from pathlib import Path
import io
import os
import sys
import subprocess
import zipfile

repo_dir = Path('/content/memact_new')
repo_dir.mkdir(exist_ok=True)
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError('Select only memact_eval_source.zip')
with zipfile.ZipFile(io.BytesIO(next(iter(uploaded.values())))) as archive:
    destination = repo_dir.resolve()
    if sum(item.file_size for item in archive.infolist()) > 50 * 1024 * 1024:
        raise ValueError('Source archive is too large')
    for item in archive.infolist():
        if not (destination / item.filename).resolve().is_relative_to(destination):
            raise ValueError('Invalid archive path')
    archive.extractall(destination)
os.chdir(repo_dir)
sys.path.insert(0, str(repo_dir))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
del uploaded
print('Memact source ready:', repo_dir)


In [ ]:
# 3. Upload your exported calibration_package.zip from your laptop
from google.colab import files
import os

print("Please select your exported calibration zip file (e.g. default_calibration_package.zip):")
uploaded = files.upload()
zip_filename = list(uploaded.keys())[0]
print(f"Uploaded: {zip_filename}")

In [ ]:
# 4. Run Cloud LoRA Fine-Tuning
!git pull
!PYTHONPATH=. python experiments/colab_train.py \n    --zip "{zip_filename}" \n    --user default \n    --epochs 15 \n    --strategy conformer_lora

In [ ]:
# 5. Download the trained personalization weights back to your laptop
from google.colab import files

result_zip = "default_personalized_weights.zip"
if os.path.exists(result_zip):
    print(f"Downloading {result_zip} to your computer...")
    files.download(result_zip)
else:
    print("File not found. Check training step logs above.")